Cleaning the raw CSVs (fixes dates, flags nulls/invalid emails/orphan rows) and writing cleaned CSVs.

In [0]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

BASE_DIR="/Volumes/assignment8/assignment8schema/assignment8volume"

customers_df=pd.read_csv(f"{BASE_DIR}/customers.csv")
products_df=pd.read_csv(f"{BASE_DIR}/products.csv")
orders_df=pd.read_csv(f"{BASE_DIR}/orders.csv")
order_items_df=pd.read_csv(f"{BASE_DIR}/order_items.csv")

issues_report={}

In [0]:
def clean_orders(df):
    df=df.copy()

    df["customer_id"]=df["customer_id"].replace("", np.nan)
    missing_customer_count=df["customer_id"].isna().sum()

    def fix_date(value):
        value=str(value).strip()
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
            try:
                return datetime.strptime(value, fmt).strftime("%Y-%m-%d %H:%M:%S")
            except ValueError:
                pass
        try:
            return datetime.strptime(value, "%d-%m-%Y").strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            return None

    df["order_date_fixed"]=df["order_date"].apply(fix_date)
    bad_dates_count=df["order_date_fixed"].isna().sum()

    df["order_date"]=df["order_date_fixed"]
    df=df.drop(columns=["order_date_fixed"])

    issues_report["orders_missing_customer_id"]=int(missing_customer_count)
    issues_report["orders_unparseable_dates"]=int(bad_dates_count)

    df=df.dropna(subset=["order_date"]).reset_index(drop=True)
    return df

def clean_products(df):
    df=df.copy()
    before=df["product_name"].copy()
    df["product_name"]=df["product_name"].str.strip().str.title()
    changed=(before.str.strip().str.title()!=before).sum()
    issues_report["products_name_normalized_count"]=int(changed)
    return df

def validate_emails(df):
    pattern=re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    invalid_mask=~df["email"].astype(str).apply(lambda e: bool(pattern.match(e)))
    invalid_ids=df.loc[invalid_mask, "customer_id"].tolist()
    issues_report["customers_invalid_email_count"]=len(invalid_ids)
    return invalid_ids

def check_referential_integrity(order_items_df, orders_df):
    valid_ids=set(orders_df["order_id"])
    bad_rows=order_items_df[~order_items_df["order_id"].isin(valid_ids)]
    issues_report["order_items_orphaned_count"]=int(len(bad_rows))
    return bad_rows

In [0]:
orders_clean=clean_orders(orders_df)
products_clean=clean_products(products_df)
invalid_email_ids=validate_emails(customers_df)
orphaned_items=check_referential_integrity(order_items_df, orders_df)

issues_report["order_items_negative_quantity_count"]=int((order_items_df["quantity"]<0).sum())
issues_report["order_items_zero_quantity_count"]=int((order_items_df["quantity"]==0).sum())
issues_report["order_items_discount_over_100_count"]=int((order_items_df["discount_percent"]>100).sum())
issues_report["order_items_future_dated_orders_count"]=int(
    pd.to_datetime(orders_clean["order_date"]).gt(pd.Timestamp.now()).sum()
)

print("=== DATA ISSUES REPORT ===")
for k, v in issues_report.items():
    print(f"{k}: {v}")

order_items_clean=order_items_df[~order_items_df.index.isin(orphaned_items.index)].reset_index(drop=True)

orders_clean.to_csv(f"{BASE_DIR}/orders_clean.csv", index=False)
products_clean.to_csv(f"{BASE_DIR}/products_clean.csv", index=False)
customers_df.to_csv(f"{BASE_DIR}/customers_clean.csv", index=False)
order_items_clean.to_csv(f"{BASE_DIR}/order_items_clean.csv", index=False)

print("Cleaned files written to:", BASE_DIR)

=== DATA ISSUES REPORT ===
orders_missing_customer_id: 163
orders_unparseable_dates: 0
products_name_normalized_count: 44
customers_invalid_email_count: 9
order_items_orphaned_count: 1
order_items_negative_quantity_count: 207
order_items_zero_quantity_count: 1
order_items_discount_over_100_count: 1
order_items_future_dated_orders_count: 0
Cleaned files written to: /Volumes/assignment8/assignment8schema/assignment8volume
